In [3]:
import secrets
import os
import pathlib
import base64

In [4]:
def gerar_chave(tamanho: int) -> bytes:
    """Gera uma chave criptograficamente segura com tamanho bytes."""
    chave_gerada_aleatoriamente = secrets.token_bytes(tamanho)
    return chave_gerada_aleatoriamente

def xor_bytes(dados: bytes, chave: bytes) -> bytes:
    """Calcula o XOR entre sequencias de mesmo comprimento."""
    if(len(dados)==len(chave)):
        # resultado_xor = bytearray()
        resultado_xor = []
        for i in range(len(dados)):
            resultado_xor.append(dados[i]^chave[i]) 
        return bytes(resultado_xor) #melhor deixar a conversão de bytes aqui já e resolver mais bunitinho do que antes
    else:
        raise ValueError("Erro! Tamanhos diferentes entre dados e chave!")


def cifrar(mensagem: bytes, chave: bytes) -> bytes:
    """Cifra mensagem usando OTP."""
    if(len(mensagem)==len(chave)):
        cifrado_calculado = xor_bytes(mensagem,chave)
        return cifrado_calculado
    else:
        raise ValueError("Erro! Tamanhos diferentes entre mensagem e chave!")

def decifrar(cifrado: bytes, chave: bytes) -> bytes:
    """Decifra um texto cifrado usando OTP."""
    if(len(cifrado)==len(chave)):
        mensagem_decifrada = xor_bytes(cifrado,chave)
        return mensagem_decifrada
    else:
        raise ValueError("Erro! Tamanhos diferentes entre cifrado e chave!")

In [21]:
import os
import hashlib

# Nomes dos arquivos de trabalho
arquivo_entrada = "materiais_laboratorio_L1_OTP/materiais_l1_otp/registros_rede_sinteticos.csv"
arquivo_chave = "materiais_laboratorio_L1_OTP/materiais_l1_otp/chave.key"
arquivo_cifrado = "materiais_laboratorio_L1_OTP/materiais_l1_otp/arquivo.cifrado"
arquivo_recuperado = "materiais_laboratorio_L1_OTP/materiais_l1_otp/arquivo.recuperado"
arquivo_SHA256 = "materiais_laboratorio_L1_OTP/materiais_l1_otp/SHA256SUMS.txt"

# Do readme:
# 2. registros_rede_sinteticos.csv
#    Pequeno conjunto de registros fictícios de rede para a Parte B. O arquivo
#    deve ser lido integralmente em modo binário, cifrado e depois recuperado.

# 1. Leitura do arquivo original com "rb" (Read Binary)
with open(arquivo_entrada, "rb") as f:
    dados_originais = f.read()

# 2. Gerar a chave (com o mesmo tamanho do arquivo lido)
chave = gerar_chave(len(dados_originais)) # como meus dados foram lidos em rb eles estão em bytes, com isso gero uma chave deste tamanho


# exercício pede para salvar a chave num arquivo separado.
# 3. Gravar a chave em disco temporariamente com "wb" (Write Binary)
with open(arquivo_chave, "wb") as f:
    f.write(chave)
    del chave # depois de escrever o arquivo chave.key eu deleto o valor da variável durante o códiguin


with open(arquivo_chave, "rb") as f:
    chave_lida_do_arquivo = f.read()

# 4. Cifrar os dados
dados_cifrados = cifrar(dados_originais, chave_lida_do_arquivo) # passo para minha função "cifrar" os meus dados em bytes e minha chave em bytes para ele realizar o xor e gerar a cifra

# exercício pede para gerar o arquivo "arquivo.cifrado"
# 5. Gravar resultados cifrados com "wb"
with open(arquivo_cifrado, "wb") as f:
    f.write(dados_cifrados)

# Releio a partir do arquivo para verificar se deu certin msm
with open(arquivo_cifrado, "rb") as f:
    dados_cifrados_lidos_arquivo = f.read()
    del dados_cifrados

# 6. Decifrar os dados
dados_decifrados = decifrar(dados_cifrados_lidos_arquivo, chave_lida_do_arquivo)

# exercício pede para gerar o arquivo "arquivo.recuperado"
# 7. Gravar resultados recuperados com "wb"
with open(arquivo_recuperado, "wb") as f:
    f.write(dados_decifrados)
    del dados_decifrados

with open(arquivo_recuperado, "rb") as f:
    dados_recuperados_lidos_arquivo = f.read()

# ==========================================
# EXIBIÇÃO DE RESULTADOS E VALIDAÇÕES
# ==========================================

# Registro dos tamanhos exigido no roteiro
print("--- Registro de Tamanhos ---")
print(f"Original:   {len(dados_originais)} bytes")
print(f"Chave:      {len(chave_lida_do_arquivo)} bytes")
print(f"Cifrado:    {len(dados_cifrados_lidos_arquivo)} bytes")
print(f"Recuperado: {len(dados_recuperados_lidos_arquivo)} bytes")

# Verificação Criptográfica com SHA-256
print("\n--- Verificação de Integridade (SHA-256) ---")

# Calculando o hash em cima dos dados que lemos lá no início
hash_original = hashlib.sha256(dados_originais).hexdigest() #hexdigest faz aparecer em hexadecimal ao invés de bytes

# Lendo o arquivo recuperado (dnv para deixar o passo a passo da ideia no código) direto do disco com "rb" para provar que salvou certo
with open(arquivo_recuperado, "rb") as f:
    dados_recuperados_lidos_arquivo = f.read()

hash_recuperado = hashlib.sha256(dados_recuperados_lidos_arquivo).hexdigest()


with open(arquivo_SHA256, "r") as f:
    for linha in f:
        if "registros_rede_sinteticos.csv" in linha:
            arquivo_sha_conferir_txt_prof, _ = linha.split()
            break



print(f"Hash Original arquivo txt:   {arquivo_sha_conferir_txt_prof}")
print(f"Hash Original:               {hash_original}")
print(f"Hash Recuperado:             {hash_recuperado}")

if hash_original == hash_recuperado == arquivo_sha_conferir_txt_prof:
    print("\nSUCESSO: Os hashes coincidem! O arquivo foi cifrado e recuperado com perfeição.")
else:
    print("\nFALHA: Os hashes são diferentes. O arquivo foi corrompido.")

# Remoção da cópia local da chave conforme o aviso
if os.path.exists(arquivo_chave):
    os.remove(arquivo_chave)
    print("\nAviso: O arquivo 'chave.key' foi removido do diretório por segurança.")

--- Registro de Tamanhos ---
Original:   897 bytes
Chave:      897 bytes
Cifrado:    897 bytes
Recuperado: 897 bytes

--- Verificação de Integridade (SHA-256) ---
Hash Original arquivo txt:   0b9ff7ac7c9d17b4541755f377b013a11dcf0d5bcb3a10536631a62ab6bfef53
Hash Original:               0b9ff7ac7c9d17b4541755f377b013a11dcf0d5bcb3a10536631a62ab6bfef53
Hash Recuperado:             0b9ff7ac7c9d17b4541755f377b013a11dcf0d5bcb3a10536631a62ab6bfef53

SUCESSO: Os hashes coincidem! O arquivo foi cifrado e recuperado com perfeição.

Aviso: O arquivo 'chave.key' foi removido do diretório por segurança.


Armazenar a chave criptográfica e o texto cifrado no mesmo local compromete totalmente a segurança do sistema, pois a força da criptografia moderna depende exclusivamente de manter a chave em segredo absoluto. Em caso de uma invasão, se um atacante conseguir acessar o servidor ou o diretório onde os arquivos estão guardados, ele obterá simultaneamente os dados protegidos e a ferramenta exata necessária para abri-los, criando o que chamamos de falha de ponto único. Como a decifração em métodos simétricos (como o One-Time Pad) é uma operação instantânea e determinística quando se tem a chave correta, o esforço de criptografar os dados torna-se completamente inútil. Na prática, é o equivalente a guardar um documento ultrassecreto em um cofre de aço impenetrável e deixar a chave colada na porta com uma fita adesiva: o cofre continua sendo tecnicamente forte, mas a proteção real foi anulada pela má gestão do acesso.